In [ ]:
!pip install jax "jax[cuda13]" transformers huggingface_hub torch einops

# UMT5 — Universal Multilingual T5

**Paper:** [arXiv:2304.09151](https://arxiv.org/abs/2304.09151) — *UniMax: Fairer and More Effective Language Sampling for Large-Scale Multilingual Pretraining* (Chung et al., Google, 2023).  
**HF source:** [google/umt5-base](https://huggingface.co/google/umt5-base)

This notebook walks through a **from-scratch JAX implementation** of UMT5-Base — a multilingual encoder-decoder used for span-filling (masked text generation).

---

### Paper Overview

UMT5 is a multilingual extension of mT5, retrained with **UniMax sampling** — a new language-sampling strategy that distributes more training budget to lower-resource languages without over-penalising high-resource ones.

| Component | Detail |
|---|---|
| **Architecture** | T5-style encoder-decoder transformer |
| **Sampling** | UniMax replaces temperature-based sampling |
| **Vocab** | 250,112 sentencepiece tokens (shared across 100+ languages) |
| **Task** | Span corruption (like T5): predict masked spans given context |

### Architecture at a glance (UMT5-Base)

```
Input tokens (B, S)
  ↓  Shared embedding (250112 → 512)
  ↓  Encoder: 8× TransformerBlock
        Self-Attention (relative pos bias) → RMSNorm → GatedGELU FFN
     Final RMSNorm → encoder_hidden_states (B, S, 512)

Decoder input tokens (B, T)
  ↓  Shared embedding (250112 → 512)
  ↓  Decoder: 8× TransformerBlock
        Causal Self-Attention (relative pos bias)
        Cross-Attention (queries=decoder, keys/values=encoder)
        GatedGELU FFN
     Final RMSNorm → scale by d_model^-0.5
  ↓  LM Head (shared weights with embedding) → logits (B, T, 250112)
```

### Imports

In [ ]:
import json
from dataclasses import dataclass
from typing import Optional

import numpy as np
import jax
import jax.numpy as jnp
import jax.nn as jnn
from jax import tree_util
import PIL.Image as Image
from safetensors.numpy import load_file
import torch
from transformers import AutoTokenizer
from einops import rearrange, repeat

**Setup:** Download the model weights and config from HuggingFace and place them at `models/umt5/` relative to your workspace root

In [ ]:
HF_REPO_ID = "google/umt5-base"
LOCAL_DIR_PATH = "workspace/models/umt5"

In this section, we download the weights of the model of interest from huggingface to use in our implementation.

In [ ]:
from huggingface_hub import snapshot_download

local_dir = snapshot_download(
    repo_id=HF_REPO_ID,
    local_dir=LOCAL_DIR_PATH,
)
print(f"Downloaded repository path: {local_dir}")

### Load Weights

UMT5 weights are loaded from the HuggingFace `google/umt5-base` checkpoint as a standard PyTorch `pytorch_model.bin`. The flat key format (`encoder.block.0.layer.0.SelfAttention.q.weight`) is parsed directly — no SafeTensors conversion needed here.

In [ ]:
WEIGHTS_PATH = f"{LOCAL_DIR_PATH}/pytorch_model.bin"
CONFIG_PATH = f"{LOCAL_DIR_PATH}/config.json"
TOKENIZER_PATH = LOCAL_DIR_PATH

hf_weights = torch.load(WEIGHTS_PATH)

### Config

UMT5-Base model dimensions — all values match the HuggingFace `config.json`:

| Field | Value | Meaning |
|---|---|---|
| `d_model` | 512 | Hidden state dimension |
| `d_ff` | 2048 | FFN intermediate dimension (4× d_model) |
| `d_kv` | 64 | Per-head key/value dimension |
| `num_heads` | 6 | Attention heads |
| `num_layers` | 8 | Encoder layers (= decoder layers) |
| `relative_attention_num_buckets` | 32 | Number of relative position buckets |
| `relative_attention_max_distance` | 128 | Max distance for relative position encoding |

In [4]:
@dataclass
class Config:
    d_model: int = 512
    d_ff: int = 2048
    d_kv: int = 64
    relative_attention_num_buckets: int = 32
    relative_attention_max_distance: int = 128
    num_layers: int = 8
    num_heads: int = 6
    num_decoder_layers: int = 8
    prompt_length: int = 216

config = Config()

### Weight Extraction

Each encoder and decoder layer has the same sub-module layout:

**Encoder block (per layer):**
- `layer.0` — Self-attention (`q, k, v, o` weights + `relative_attention_bias` + `layer_norm`)
- `layer.1` — GatedGELU FFN (`wi_0, wi_1, wo` + `layer_norm`)

**Decoder block (per layer):**
- `layer.0` — Causal self-attention (same as encoder, with `is_decoder=True`)
- `layer.1` — Cross-attention / EncDecAttention (`q, k, v, o` + `layer_norm`, **no** relative position bias)
- `layer.2` — GatedGELU FFN (identical structure to encoder FFN)

The `lm_head` weight is **untied** from the shared embedding in this checkpoint.

In [ ]:
def get_w(
    name: str,
    transpose: bool = False
) -> jax.Array:
    val = hf_weights.pop(name)
    w = jnp.array(val.detach().cpu(), dtype=jnp.bfloat16)
    return w.T if transpose else w

encoder_blocks = []
for i in range(8):
    block = [
        {
            'q': get_w(f'encoder.block.{i}.layer.0.SelfAttention.q.weight'),
            'k': get_w(f'encoder.block.{i}.layer.0.SelfAttention.k.weight'),
            'v': get_w(f'encoder.block.{i}.layer.0.SelfAttention.v.weight'),
            'o': get_w(f'encoder.block.{i}.layer.0.SelfAttention.o.weight'),
            'relative_attention_bias': get_w(f'encoder.block.{i}.layer.0.SelfAttention.relative_attention_bias.weight'),
            'norm': get_w(f'encoder.block.{i}.layer.0.layer_norm.weight'),
        },
        {
            'w1': get_w(f'encoder.block.{i}.layer.1.DenseReluDense.wi_0.weight'),
            'w2': get_w(f'encoder.block.{i}.layer.1.DenseReluDense.wi_1.weight'),
            'w3': get_w(f'encoder.block.{i}.layer.1.DenseReluDense.wo.weight'),
            'norm': get_w(f'encoder.block.{i}.layer.1.layer_norm.weight'),
        }
    ]
    encoder_blocks.append(block)

decoder_blocks = []
for i in range(8):
    block = [
        {
            'q': get_w(f'decoder.block.{i}.layer.0.SelfAttention.q.weight'),
            'k': get_w(f'decoder.block.{i}.layer.0.SelfAttention.k.weight'),
            'v': get_w(f'decoder.block.{i}.layer.0.SelfAttention.v.weight'),
            'o': get_w(f'decoder.block.{i}.layer.0.SelfAttention.o.weight'),
            'relative_attention_bias': get_w(f'decoder.block.{i}.layer.0.SelfAttention.relative_attention_bias.weight'),
            'norm': get_w(f'decoder.block.{i}.layer.0.layer_norm.weight'),
        },
        {
            'q': get_w(f'decoder.block.{i}.layer.1.EncDecAttention.q.weight'),
            'k': get_w(f'decoder.block.{i}.layer.1.EncDecAttention.k.weight'),
            'v': get_w(f'decoder.block.{i}.layer.1.EncDecAttention.v.weight'),
            'o': get_w(f'decoder.block.{i}.layer.1.EncDecAttention.o.weight'),
            'norm': get_w(f'decoder.block.{i}.layer.1.layer_norm.weight'),
        },
        {
            'w1': get_w(f'decoder.block.{i}.layer.2.DenseReluDense.wi_0.weight'),
            'w2': get_w(f'decoder.block.{i}.layer.2.DenseReluDense.wi_1.weight'),
            'w3': get_w(f'decoder.block.{i}.layer.2.DenseReluDense.wo.weight'),
            'norm': get_w(f'decoder.block.{i}.layer.2.layer_norm.weight'),
        }
    ]
    decoder_blocks.append(block)

m = {
    'shared': get_w('shared.weight'),
    'encoder_embed': get_w('encoder.embed_tokens.weight'),
    'encoder_blocks': encoder_blocks,
    'encoder_norm': get_w('encoder.final_layer_norm.weight'),
    'decoder_embed': get_w('decoder.embed_tokens.weight'),
    'decoder_blocks': decoder_blocks,
    'decoder_norm': get_w('decoder.final_layer_norm.weight'),
    'lm_head': get_w('lm_head.weight'),
}

In [6]:
hf_weights.keys()

dict_keys([])

### Tokenizer & Input

UMT5 uses a **SentencePiece** tokenizer with 250,112 tokens shared across 100+ languages. It's the same vocabulary as mT5.

The span-corruption task uses special tokens like `<extra_id_0>`, `<extra_id_1>`, … as sentinel markers for masked spans. At inference, we give the model a partially-masked sentence and let it autoregressively fill in the blanks.

The encoder input is zero-padded to a fixed `prompt_length` (216 tokens here). A padding mask is built to prevent the model from attending to pad positions.

In [7]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(TOKENIZER_PATH)
prompt = "The capital of <extra_id_1> is Paris."
input_ids = tokenizer(prompt, return_tensors='np')['input_ids']
input_ids

The tokenizer you are loading from '../../.models/umt5' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.


array([[   517,   5801,    329, 256298,    339,   2989,    274,      1]])

In [8]:
input_length = input_ids.shape[-1]
padding_length = config.prompt_length - input_length
if padding_length > 0:
    input_ids = jnp.pad(input_ids, ((0, 0), (0, padding_length)), constant_values=tokenizer.pad_token_id)

input_ids.shape

(1, 216)

### Attention Masking

Two masks are needed:

**Encoder padding mask** — prevents the encoder from attending to `<pad>` tokens. Pad positions get a large negative value (`-inf`) added to the attention logits before softmax, effectively zeroing out their attention weights.

**Decoder causal mask** — a lower-triangular mask that prevents each decoder token from attending to future positions. Token `t` can only see tokens `0…t`. This is essential for autoregressive generation — the model must not "cheat" by looking ahead.

In [9]:
def attention_mask(
    input_ids: jax.Array  # (B, S)
) -> jax.Array:           # (B, 1, 1, S)
    padding_mask = (input_ids != tokenizer.pad_token_id)
    padding_mask = rearrange(padding_mask, 'b s -> b 1 1 s')
    encoder_mask = (~padding_mask).astype(jnp.float32) * jnp.finfo(jnp.float32).min
    return encoder_mask

def causal_attention_mask(
    seq_len: int
) -> jax.Array:  # (1, 1, S, S)
    mask = jnp.tril(jnp.ones((seq_len, seq_len), dtype=jnp.float32))
    mask = (1.0 - mask) * jnp.finfo(jnp.float32).min
    mask = rearrange(mask, 'q k -> 1 1 q k')
    return mask

### RMS Norm

UMT5 (like T5 v1.1 and most modern LLMs) uses **Root Mean Square Layer Norm** instead of standard LayerNorm — no mean subtraction, just RMS scaling:

$$\text{RMSNorm}(x) = \frac{x}{\sqrt{\frac{1}{D}\sum x_i^2 + \epsilon}} \cdot w$$

This is applied **before** each sub-layer (pre-norm), not after — a design choice that improves training stability.

In [10]:
def rms_norm(
    x: jax.Array,   # (..., D)
    w: jax.Array,   # (D,)
    eps: float = 1e-6
) -> jax.Array:     # (..., D)
    norm_x = x / jnp.sqrt(jnp.mean(x ** 2, axis=-1, keepdims=True) + eps)
    return norm_x * w

### Relative Position Bias

T5/UMT5 uses **bucket-based relative position bias** instead of absolute positional embeddings. The idea: map the relative distance between every query-key pair into one of `num_buckets` buckets, then look up a learned scalar bias per bucket per head.

**Bucket allocation (encoder, bidirectional):**
- Half the buckets cover **exact** small distances (0 to `max_exact - 1`)
- The other half cover **logarithmically-spaced** larger distances

**Decoder (causal):** Only negative relative distances (past positions) are used, so all buckets cover the one-sided range `[0, max_distance]` without the sign-split.

`relative_position_bucket` maps the `(Q, K)` relative distance matrix → bucket indices.  
`compute_bias` looks up the learned bias table and returns a `(1, num_heads, Q, K)` additive bias for the attention logits.

In [11]:
def relative_position_bucket(
    rel_pos: jax.Array,     # (Q, K)
    is_decoder: bool = False
) -> jax.Array:             # (Q, K)
    if not is_decoder:
        num_buckets = config.relative_attention_num_buckets // 2
        rel_buckets = (rel_pos > 0).astype(jnp.int32) * num_buckets
        rel_pos = jnp.abs(rel_pos)
    else:
        num_buckets = config.relative_attention_num_buckets
        rel_buckets = 0
        rel_pos = -jnp.minimum(rel_pos, jnp.zeros_like(rel_pos))

    max_exact = num_buckets // 2
    is_small = rel_pos < max_exact

    rel_pos_large = max_exact + (
        jnp.log(rel_pos.astype(jnp.float32) / max_exact)
        / jnp.log(config.relative_attention_max_distance / max_exact)
        * (num_buckets - max_exact)
    ).astype(jnp.int32)
    rel_pos_large = jnp.minimum(rel_pos_large, num_buckets - 1)

    rel_buckets = rel_buckets + jnp.where(is_small, rel_pos.astype(jnp.int32), rel_pos_large)
    return rel_buckets

In [12]:
def compute_bias(
    q_len: int,
    k_len: int,
    relative_attention_bias: jax.Array,  # (num_buckets, num_heads)
    is_decoder: bool = False
) -> jax.Array:                          # (1, num_heads, Q, K)
    ctx_pos = jnp.arange(q_len, dtype=jnp.int32)[:, None]
    mem_pos = jnp.arange(k_len, dtype=jnp.int32)[None, :]
    rel_pos = mem_pos - ctx_pos
    rel_pos_bkt = relative_position_bucket(rel_pos, is_decoder)

    values = relative_attention_bias[rel_pos_bkt]
    values = rearrange(values, 'q k h -> 1 h q k')
    return values

### Cross-Attention

`umt5_cross_attention` handles the encoder-decoder attention in each decoder block. Queries come from the **decoder** hidden states, while keys and values come from the **encoder** output — this is how the decoder "reads" the encoded input.

Unlike self-attention, there is **no relative position bias** here — only the padding mask from the encoder is applied. The spatial relationship between encoder and decoder tokens is not meaningful in the same way as within a single sequence.

In [13]:
def umt5_cross_attention(
    x: jax.Array,                     # (B, Q, D)
    encoder_hidden_states: jax.Array, # (B, K, D)
    weights: dict,
    attention_mask: jax.Array = None  # (B, 1, 1, K)
) -> jax.Array:                       # (B, Q, D)
    attn = weights

    q = x @ attn['q'].T
    k = encoder_hidden_states @ attn['k'].T
    v = encoder_hidden_states @ attn['v'].T

    q_len = q.shape[-2]
    k_len = k.shape[-2]

    q = rearrange(q, 'b i (n c) -> b i n c', n=config.num_heads)
    k = rearrange(k, 'b j (n c) -> b j n c', n=config.num_heads)
    v = rearrange(v, 'b j (n c) -> b j n c', n=config.num_heads)

    position_bias = jnp.zeros((1, config.num_heads, q_len, k_len), dtype=q.dtype)

    if attention_mask is not None:
        position_bias = position_bias + attention_mask

    attn_out = jnp.einsum('binc, bjnc -> bnij', q, k)
    attn_out = attn_out + position_bias
    attn_out = jnn.softmax(attn_out, axis=-1)

    out = jnp.einsum('bnij, bjnc -> binc', attn_out, v)
    out = rearrange(out, 'b i n c -> b i (n c)')
    out = out @ attn['o'].T
    return out

### Self-Attention

`umt5_attention` handles both encoder self-attention (bidirectional) and decoder causal self-attention — the `is_decoder` flag controls which relative position bucket scheme is used.

The attention pattern follows standard multi-head scaled dot-product attention, with the relative position bias **added to the raw logits** before softmax:

$$\text{Attn}(Q, K, V) = \text{softmax}\!\left(\frac{QK^\top}{\sqrt{d_{kv}}} + B_{\text{rel}}\right) V$$

Note UMT5 does **not** scale Q by $\frac{1}{\sqrt{d_{kv}}}$ explicitly — the relative bias absorbs position-sensitive calibration.

In [14]:
def umt5_attention(
    x: jax.Array,                     # (B, S, D)
    weights: dict,
    is_decoder: bool = False,
    attention_mask: jax.Array = None  # (1, 1, S, S) or (B, 1, 1, S)
) -> jax.Array:                       # (B, S, D)
    attn = weights

    q = x @ attn['q'].T
    k = x @ attn['k'].T
    v = x @ attn['v'].T

    q_len = q.shape[-2]
    k_len = k.shape[-2]

    q = rearrange(q, 'b i (n c) -> b i n c', n=config.num_heads)
    k = rearrange(k, 'b j (n c) -> b j n c', n=config.num_heads)
    v = rearrange(v, 'b j (n c) -> b j n c', n=config.num_heads)

    if 'relative_attention_bias' in attn:
        position_bias = compute_bias(q_len, k_len, attn['relative_attention_bias'], is_decoder=is_decoder)
    else:
        position_bias = jnp.zeros((1, config.num_heads, q_len, k_len), dtype=q.dtype)

    if attention_mask is not None:
        position_bias = position_bias + attention_mask

    attn_out = jnp.einsum('binc, bjnc -> bnij', q, k)
    attn_out = attn_out + position_bias
    attn_out = jnn.softmax(attn_out, axis=-1)

    out = jnp.einsum('bnij, bjnc -> binc', attn_out, v)
    out = rearrange(out, 'b i n c -> b i (n c)')

    out = out @ attn['o'].T
    return out

### Encoder Block

Each encoder layer follows a **pre-norm** residual pattern:

```
x → RMSNorm → Self-Attention → (+) → RMSNorm → GatedGELU FFN → (+)
────────────────────────────────┘ ──────────────────────────────┘
```

The FFN is a **Gated Linear Unit (GLU) with GELU**, sometimes written as SwiGLU/GeGLU. Two parallel projections are computed and their outputs are multiplied element-wise before the final projection:

```
gate = GELU(x @ w1.T)
value = x @ w2.T
hidden = gate * value
out = hidden @ w3.T
```

This gating mechanism gives the FFN more expressive power than a standard two-layer MLP for the same parameter count.

In [15]:
def encoder_block(
    hidden_states: jax.Array,  # (B, S, D)
    block: list,
    attention_mask: jax.Array  # (B, 1, 1, S)
) -> jax.Array:                # (B, S, D)
    attn = block[0]
    ffn = block[1]

    residual = hidden_states
    hidden_states = rms_norm(hidden_states, attn['norm'])
    attention_output = umt5_attention(hidden_states, attn, is_decoder=False, attention_mask=attention_mask)
    hidden_states = residual + attention_output

    residual = hidden_states
    hidden_states = rms_norm(hidden_states, ffn['norm'])

    gate = jnn.gelu(hidden_states @ ffn['w1'].T, approximate=True)
    value = hidden_states @ ffn['w2'].T

    hidden_states = gate * value
    hidden_states = hidden_states @ ffn['w3'].T

    hidden_states = residual + hidden_states

    return hidden_states

### Decoder Block

The decoder block runs **three sub-layers** per layer (vs two in the encoder):

```
x → RMSNorm → Causal Self-Attn → (+)
            → RMSNorm → Cross-Attn (← encoder output) → (+)
            → RMSNorm → GatedGELU FFN → (+)
```

- **Causal self-attention** uses a triangular mask so token `t` only attends to `0…t`
- **Cross-attention** has no causal mask — each decoder token can freely attend to any encoder token, gated only by the encoder padding mask
- The FFN is identical to the encoder FFN

In [16]:
def decoder_block(
    hidden_states: jax.Array,         # (B, T, D)
    encoder_hidden_states: jax.Array, # (B, S, D)
    block: list,
    causal_mask: jax.Array,           # (1, 1, T, T)
    attention_mask: jax.Array         # (B, 1, 1, S)
) -> jax.Array:                       # (B, T, D)
    self_attn = block[0]
    enc_dec_attn = block[1]
    ffn = block[2]

    residual = hidden_states
    hidden_states = rms_norm(hidden_states, self_attn['norm'])
    attention_output = umt5_attention(hidden_states, self_attn, is_decoder=True, attention_mask=causal_mask)
    hidden_states = residual + attention_output

    residual = hidden_states
    hidden_states = rms_norm(hidden_states, enc_dec_attn['norm'])
    attention_output = umt5_cross_attention(hidden_states, encoder_hidden_states, enc_dec_attn, attention_mask=attention_mask)
    hidden_states = residual + attention_output

    residual = hidden_states
    hidden_states = rms_norm(hidden_states, ffn['norm'])

    gate = jnn.gelu(hidden_states @ ffn['w1'].T, approximate=True)
    value = hidden_states @ ffn['w2'].T

    hidden_states = gate * value
    hidden_states = hidden_states @ ffn['w3'].T

    hidden_states = residual + hidden_states

    return hidden_states

### Full Forward Pass (Greedy Decoding)

UMT5 generates tokens **autoregressively** — one token at a time:

```
1. Encode full input once:
   input_ids → embedding → 8× encoder_block → RMSNorm → encoder_hidden_states

2. Decode token by token:
   decoder_input_ids = [<pad>]   ← BOS token
   loop:
       decoder_tokens → embedding → 8× decoder_block → RMSNorm
       → scale by d_model^-0.5
       → logits[:, -1, :] @ lm_head.T   ← only the last position
       → argmax → next_token
       → append to decoder_input_ids
       → stop if EOS
```

The `d_model ** -0.5` scaling before the LM head is T5's way of compensating for the lack of softmax temperature in the vocabulary projection — it keeps the logit scale stable regardless of hidden dimension size.

In [17]:
max_new_tokens = 20

embedding = m['encoder_embed'][input_ids]
padding_mask = attention_mask(input_ids)

encoder_hidden_states = embedding
for block in m['encoder_blocks']:
    encoder_hidden_states = encoder_block(encoder_hidden_states, block, padding_mask)
encoder_hidden_states = rms_norm(encoder_hidden_states, m['encoder_norm']) 

decoder_input_ids = jnp.array([[tokenizer.pad_token_id]], dtype=jnp.int32)
generated = []

for _ in range(max_new_tokens):

    decoder_embed = m['decoder_embed'][decoder_input_ids]
    dec_seq_len = decoder_embed.shape[1]
    causal_mask = causal_attention_mask(dec_seq_len)
    
    hidden_states = decoder_embed
    for block in m['decoder_blocks']:
        hidden_states = decoder_block(hidden_states, encoder_hidden_states, block, causal_mask, padding_mask)
    hidden_states = rms_norm(hidden_states, m['decoder_norm'])

    scale = config.d_model ** -0.5
    hidden_states = hidden_states * scale

    next_token_logits = hidden_states[:, -1, :] @ m['lm_head'].T
    next_token_id = jnp.argmax(next_token_logits, axis=-1)
    generated.append(int(next_token_id[0]))
    
    print(tokenizer.decode(next_token_id))

    if int(next_token_id[0]) == tokenizer.eos_token_id:
        break

    decoder_input_ids = jnp.concatenate(
        [decoder_input_ids, next_token_id[:, None]], axis=1
    )

print("Full output:", tokenizer.decode(generated))

<extra_id_0>
is
Paris
,
the
capital
of
France
is
Paris
is
Paris
is
Paris
is
Paris
<extra_id_1>
France
is
Paris
Full output: <extra_id_0> is Paris, the capital of France is Paris is Paris is Paris is Paris<extra_id_1> France is Paris
